# Using pdfplumber with the Resume Dataset

This notebook demonstrates how to use pdfplumber to extract text from the PDF resume dataset.

## 2. Basic Usage - Read a Single PDF

In [2]:
import pdfplumber
import os

# Path to your PDF dataset
data_path = "data/resume-dataset/data/data"

# Example: Read a single PDF
pdf_path = os.path.join(data_path, "ACCOUNTANT/10554236.pdf")

with pdfplumber.open(pdf_path) as pdf:
    # Get number of pages
    print(f"Number of pages: {len(pdf.pages)}")
    
    # Extract text from all pages
    full_text = ""
    for page in pdf.pages:
        full_text += page.extract_text() or ""
    
    print(full_text)

Number of pages: 5
ACCOUNTANT
Summary
Financial Accountant specializing in financial planning, reporting and analysis within the Department of Defense.
Highlights
Account reconciliations
Results-oriented Accounting operations professional
Financial reporting Analysis of financial systems
Critical thinking ERP (Enterprise Resource Planning) software.
Excellent facilitator
Accomplishments
Served on a tiger team which identified and resolved General Ledger postings in DEAMS totaling $360B in accounting adjustments. This allowed
for the first successful fiscal year-end close for 2012.
In collaboration with DFAS Europe, developed an automated tool that identified duplicate obligations. This tool allowed HQ USAFE to
deobligate over $5M in duplicate obligations.
Experience
Company Name July 2011 to November 2012 Accountant
City , State
Enterprise Resource Planning Office (ERO)
In this position as an Accountant assigned to the Defense Enterprise Accounting and Management System (DEAMS) ERO I w

## 3. Process Multiple PDFs by Category

In [3]:
import pdfplumber
import os
from pathlib import Path

data_path = Path("data/resume-dataset/data/data")

def extract_text_from_pdf(pdf_path):
    """Extract all text from a PDF file."""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = ""
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
            return text.strip()
    except Exception as e:
        print(f"Error processing {pdf_path}: {e}")
        return None

# Process all PDFs in a category
category = "ENGINEERING"
category_path = data_path / category

resumes = []
for pdf_file in category_path.glob("*.pdf"):
    text = extract_text_from_pdf(pdf_file)
    if text:
        resumes.append({
            "id": pdf_file.stem,
            "category": category,
            "text": text
        })

print(f"Processed {len(resumes)} resumes from {category}")

Processed 118 resumes from ENGINEERING


## 4. Process Entire Dataset into DataFrame

In [4]:
import pdfplumber
import pandas as pd
from pathlib import Path
from tqdm import tqdm

data_path = Path("data/resume-dataset/data/data")

def extract_text_from_pdf(pdf_path):
    try:
        with pdfplumber.open(pdf_path) as pdf:
            return "\n".join(
                page.extract_text() or "" for page in pdf.pages
            ).strip()
    except Exception as e:
        return None

# Get all categories (subdirectories)
categories = [d.name for d in data_path.iterdir() if d.is_dir()]

# Process all PDFs
records = []
for category in tqdm(categories, desc="Categories"):
    category_path = data_path / category
    for pdf_file in category_path.glob("*.pdf"):
        text = extract_text_from_pdf(pdf_file)
        records.append({
            "id": pdf_file.stem,
            "category": category,
            "text": text
        })

# Create DataFrame
df = pd.DataFrame(records)
print(f"Total resumes: {len(df)}")
df.head()

Categories: 100%|██████████| 24/24 [06:19<00:00, 15.80s/it]

Total resumes: 2484


,id,category,text
0,37201447,AGRICULTURE,ADULT EDUCATION INSTRUCTOR\nSummary\nSeasoned ...
1,12674256,AGRICULTURE,FINANCIAL SALES CONSULTANT\nProfessional Summa...
2,29968330,AGRICULTURE,EXTENSION METHODOLOGIST\nProfile\nSelf-motivat...
3,81042872,AGRICULTURE,RESEARCH SCIENTIST\nSummary\nHighly motivated ...
4,20006992,AGRICULTURE,"FRONT DESK CLERK (FEE BASIS, JOHN D DINGELL VA..."


## 5. Extract Tables from PDFs (if any)

In [5]:
import pdfplumber

pdf_path = "data/resume-dataset/data/data/ACCOUNTANT/10554236.pdf"

with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages):
        tables = page.extract_tables()
        if tables:
            print(f"Page {i+1} has {len(tables)} table(s)")
            for table in tables:
                for row in table:
                    print(row)

## 6. Advanced: Extract Words with Position Data

In [6]:
import pdfplumber

pdf_path = "data/resume-dataset/data/data/ACCOUNTANT/10554236.pdf"

with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    words = page.extract_words()
    
    # Show first 10 words with their positions
    for word in words[:10]:
        print(f"Text: {word['text']:20} | Position: ({word['x0']:.1f}, {word['top']:.1f})")

Text: ACCOUNTANT           | Position: (33.3, 28.3)
Text: Summary              | Position: (33.3, 40.3)
Text: Financial            | Position: (33.3, 61.9)
Text: Accountant           | Position: (67.5, 61.9)
Text: specializing         | Position: (111.3, 61.9)
Text: in                   | Position: (153.9, 61.9)
Text: financial            | Position: (162.3, 61.9)
Text: planning,            | Position: (193.5, 61.9)
Text: reporting            | Position: (227.8, 61.9)
Text: and                  | Position: (262.6, 61.9)


## Key pdfplumber Methods Reference

| Method | Description |
|--------|-------------|
| `pdf.pages` | List of all pages |
| `page.extract_text()` | Extract text from page |
| `page.extract_tables()` | Extract tables as lists |
| `page.extract_words()` | Get word-level details with positions |
| `page.chars` | Character-level data |
| `page.images` | Image metadata |

**Dataset Info:**
- **2,484 PDFs** organized in **24 job categories**
- Categories: ACCOUNTANT, ADVOCATE, AGRICULTURE, APPAREL, ARTS, AUTOMOBILE, AVIATION, BANKING, BPO, BUSINESS-DEVELOPMENT, CHEF, CONSTRUCTION, CONSULTANT, DESIGNER, DIGITAL-MEDIA, ENGINEERING, FINANCE, FITNESS, HEALTHCARE, HR, INFORMATION-TECHNOLOGY, PUBLIC-RELATIONS, SALES, TEACHER